In [1]:
import numpy as np
from __future__ import annotations
import os, json
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from typing import List, Dict, Any

from __future__ import annotations


In [2]:
# Define per-dataset annotation/removal times (seconds)
ANNOTATION_TIMES = {
    "tomatoes": {"add": 4.84, "remove": 2.05},
    "apples": {"add": 5.46, "remove": 2.05}
}
# Default removal time (seconds)
DEFAULT_ADD_TIME = 5.15  # seconds
DEFAULT_REMOVE_TIME = 2.05  # seconds   

# Define review time per bounding box (seconds)
REVIEW_TIME_PER_BOX = 0.33  # seconds

# Define hourly compute cost ($/hour)
HOURLY_COMPUTE_COST = 2.03081  # $/hour

# Define IoU thresholds for evaluation
EVAL_IOU_THRESHOLDS = [0.5,0.7,0.9]

In [3]:
def _xywh_to_xyxy(b):
    x, y, w, h = b
    return (x, y, x + w, y + h)

def _iou(b1, b2):
    x1,y1,x2,y2 = _xywh_to_xyxy(b1)
    x1g, y1g, x2g, y2g = _xywh_to_xyxy(b2)
    ix1, iy1 = max(x1, x1g), max(y1, y1g)
    ix2, iy2 = min(x2, x2g), min(y2, y2g)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x2g - x1g) * (y2g - y1g)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def compute_pr_manual(
    gt_annotations,         # list of dicts: {"image_id", "category_id", "bbox"}
    pred_annotations,       # list of dicts: {"image_id", "category_id", "bbox", "score"}
    iou_thr=0.50,
    image_ids_eval=None     # optional: restrict to these GT image_ids
):
    """
    Returns: dict with TP, FP, FN, precision, recall
    Notes:
      - Greedy one-to-one matching per (image_id, category_id)
      - All predictions considered (no maxDets)
      - All gt considered (no area/crowd filtering)
    """

    # group GT and predictions by (image_id, category_id)
    gts = defaultdict(list)
    dts = defaultdict(list)

    if image_ids_eval is not None:
        image_ids_eval = set(image_ids_eval)

    for g in gt_annotations:
        if image_ids_eval is not None and g["image_id"] not in image_ids_eval:
            continue
        gts[(g["image_id"], g["category_id"])].append({"bbox": g["bbox"], "matched": False})

    for d in pred_annotations:
        if image_ids_eval is not None and d["image_id"] not in image_ids_eval:
            continue
        dts[(d["image_id"], d["category_id"])].append({"bbox": d["bbox"], "score": float(d.get("score", 1.0))})

    tp = 0
    fp = 0
    fn = 0

    # iterate over all keys present in either GT or preds
    keys = set(gts.keys()) | set(dts.keys())
    for key in keys:
        gt_list = gts.get(key, [])
        dt_list = dts.get(key, [])

        # sort detections by score desc
        dt_list.sort(key=lambda x: x["score"], reverse=True)

        gt_matched = [False] * len(gt_list)

        # greedy matching
        for det in dt_list:
            best_iou = 0.0
            best_j = -1
            for j, gt in enumerate(gt_list):
                if gt_matched[j]:
                    continue
                iou = _iou(det["bbox"], gt["bbox"])
                if iou >= iou_thr and iou > best_iou:
                    best_iou = iou
                    best_j = j
            if best_j >= 0:
                gt_matched[best_j] = True
                tp += 1
            else:
                fp += 1

        # any unmatched GT are FN
        fn += sum(1 for m in gt_matched if not m)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
    }


In [4]:
def parse_filename(stem: str) -> tuple[str, str, str] | None:
    """
    Parse dataset, subset, model from a filename stem.

    Expected: dataset_subset_model_predictions
    Falls back to splitting at 'predictions' token.
    """
    parts = stem.split("_")

    # Standard pattern: at least 3 parts and last token is 'predictions'
    if len(parts) >= 3 and parts[-1] == "predictions":
        dataset = parts[0]
        subset = parts[1]
        model = "_".join(parts[2:-1])  # between subset and 'predictions'
        return dataset, subset, model

    # Fallback: try to find explicit 'predictions' token
    try:
        pred_idx = parts.index("predictions")
        core = parts[:pred_idx]
    except ValueError:
        core = parts

    if len(core) < 3:
        return None

    dataset = core[0]
    subset = core[1]
    model = "_".join(core[2:])
    return dataset, subset, model

In [5]:
def make_empty_eval(
    dataset_name: str,
    subset_name: str,
    model_name: str,
    iou_type: str,
    iou_thresholds: List[float],
) -> Dict[str, Any]:
    metric_keys = [
        "AP@[.50:.95]", "AP@0.50", "AP@0.75",
        "AP_small", "AP_medium", "AP_large",
        "AR@1", "AR@10", "AR@100",
        "AR_small", "AR_medium", "AR_large",
    ]
    coco_metrics = {k: 0.0 for k in metric_keys}

    # Build conditional efficiency metric dicts
    efficiency_metrics = {
        f"efficiency_metrics@iou_{t:.2f}": {
            "bbox_additions": 0,
            "bbox_removals": 0,
            "precision": 0.0,
            "recall": 0.0,
            "total_annotation_time_s": 0.0,
            "annotation_time_per_bbox_s": 0.0,
            "annotation_time_manual_pct": 0.0,
            "annotation_time_machine_pct": 0.0,
            "total_cost_usd": 0.0,
            "cost_per_bbox_usd": 0.0,
        }
        for t in iou_thresholds
    }

    return {
        "info": {
            "timestamp": "",
            "dataset_name": dataset_name,
            "subset_name": subset_name,
            "model_name": model_name,
            "iou_type": iou_type,
            "num_predicted_bbox": 0,
            "num_gt_bbox": 0,
            "num_eval_images": 0,
            "num_eval_categories": 0,
        },
        "coco_metrics": coco_metrics,
        **efficiency_metrics,
    }


In [6]:
def evaluate_predictions(
    predictions_dir: str | Path,
    iou_thresholds: List[float],
    iou_type: str = "bbox",
) -> List[Dict[str, Any]]:
    """
    Evaluate COCO-style prediction files in a directory.

    Expects prediction files named:
        {dataset}_{subset}_{model}_predictions.json

    Writes evaluations to:
        ../Evaluations/{PREDICTIONS_DIR_NAME}/{dataset}_{subset}_{model}_evaluation.json

    Requirements:
      - Ground truth at: ../Data/{dataset}/annotations/instances_{subset}.json

    Returns:
      - a list of evaluation summaries (one per prediction file)
    """

    # Prepare output directory
    predictions_dir = Path(predictions_dir)
    out_root = Path("..") / "Evaluations" / predictions_dir.name
    out_root.mkdir(parents=True, exist_ok=True)

    # Find prediction JSON files
    prediction_jsons = sorted(predictions_dir.glob("*_predictions.json"))
    if not prediction_jsons:
        print(f"No prediction files found in {predictions_dir}")
        return []

    results: List[Dict[str, Any]] = []

    for pred_json in prediction_jsons:
        # Infer dataset, subset, model from filename
        parsed = parse_filename(pred_json.stem)
        if parsed is None:
            print(f"Skip unrecognized filename pattern: {pred_json.name}")
            continue
        dataset_name, subset_name, model_name = parsed

        # Locate GT file
        gt_path = Path("..") / "Data" / dataset_name / "annotations" / f"instances_{subset_name}.json"
        if not gt_path.exists():
            print(f"Missing GT: {gt_path}")
            continue

        # Load COCO GT
        cocoGt = COCO(str(gt_path))

        # Ensure every GT annotation has an `iscrowd` field (default 0)
        for ann in cocoGt.dataset.get("annotations", []):
            ann.setdefault("iscrowd", 0)
        # Rebuild indices after modifying the dataset
        cocoGt.createIndex()

        # Load predictions
        with open(pred_json, "r", encoding="utf-8") as f:
            preds = json.load(f)

        # Map GT and prediction images by filename
        gt_name_to_id = {
            os.path.basename(im["file_name"]): im["id"]
            for im in cocoGt.dataset["images"]
        }

        pred_images = preds.get("images", [])
        pred_id_to_name = {
            im["id"]: os.path.basename(im["file_name"])
            for im in pred_images
        }

        # Image set: all images listed in preds["images"] that exist in GT
        img_ids_eval = sorted({
            gt_name_to_id[os.path.basename(im["file_name"])]
            for im in pred_images
            if os.path.basename(im["file_name"]) in gt_name_to_id
        })

        # If there is no overlap between GT and predictions, write zeros and continue
        if not img_ids_eval:
            eval_summary = make_empty_eval(dataset_name, subset_name, model_name, iou_type, iou_thresholds)
            eval_json = out_root / f"{dataset_name}_{subset_name}_{model_name}_evaluation.json"
            with open(eval_json, "w", encoding="utf-8") as f:
                json.dump(eval_summary, f, indent=2)
            results.append(eval_summary)
            continue

        # ------------------------------------------------------------------
        # Categories: use GT category IDs directly; ignore names completely
        # ------------------------------------------------------------------
        cat_ids_eval = sorted(c["id"] for c in cocoGt.dataset["categories"])

        # ------------------------------------------------------------------
        # Remap detections: only image IDs are remapped; category_ids used as-is
        # ------------------------------------------------------------------
        remapped: List[Dict[str, Any]] = []
        valid_cat_ids = set(cat_ids_eval)

        for ann in preds.get("annotations", []):
            pred_name = pred_id_to_name.get(ann["image_id"])
            if not pred_name:
                continue

            gt_img_id = gt_name_to_id.get(pred_name)
            if gt_img_id is None:
                continue

            cat_id = ann["category_id"]  # must already match GT category_id

            # Optionally drop detections with invalid category_id
            if cat_id not in valid_cat_ids:
                continue

            x, y, w, h = ann["bbox"]

            remapped.append({
                "image_id": gt_img_id,
                "category_id": cat_id,
                "bbox": [float(x), float(y), float(w), float(h)],
                "score": float(ann.get("score", 0.0)),  # default score 0.0 if missing
            })

        # ACTUAL EVALUATION
        if len(remapped) > 0:
            cocoDt = cocoGt.loadRes(remapped)
            cocoEval = COCOeval(cocoGt, cocoDt, iouType=iou_type)
            cocoEval.params.imgIds = img_ids_eval
            cocoEval.params.catIds = cat_ids_eval

            cocoEval.evaluate()
            cocoEval.accumulate()
            cocoEval.summarize()

            coco_metrics = {
                "AP@[.50:.95]": cocoEval.stats[0],
                "AP@0.50":      cocoEval.stats[1],
                "AP@0.75":      cocoEval.stats[2],
                "AP_small":     cocoEval.stats[3],
                "AP_medium":    cocoEval.stats[4],
                "AP_large":     cocoEval.stats[5],
                "AR@1":         cocoEval.stats[6],
                "AR@10":        cocoEval.stats[7],
                "AR@100":       cocoEval.stats[8],
                "AR_small":     cocoEval.stats[9],
                "AR_medium":    cocoEval.stats[10],
                "AR_large":     cocoEval.stats[11],
            }
        else:
            # no predictions for these images → coco_metrics are all zero
            coco_metrics = {
                "AP@[.50:.95]": 0.0,
                "AP@0.50":      0.0,
                "AP@0.75":      0.0,
                "AP_small":     0.0,
                "AP_medium":    0.0,
                "AP_large":     0.0,
                "AR@1":         0.0,
                "AR@10":        0.0,
                "AR@100":       0.0,
                "AR_small":     0.0,
                "AR_medium":    0.0,
                "AR_large":     0.0,
            }

        # Additional metrics
        num_gt_bbox = sum(
            1 for a in cocoGt.dataset["annotations"]
            if a["image_id"] in img_ids_eval
        )
        num_predicted_bbox = len(remapped)

        # Initialize eval_summary (with empty efficiency_metrics per IoU)
        eval_summary = make_empty_eval(
            dataset_name=dataset_name,
            subset_name=subset_name,
            model_name=model_name,
            iou_type=iou_type,
            iou_thresholds=iou_thresholds,
        )

        # Fill info and coco_metrics
        eval_summary["info"].update({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "num_predicted_bbox": num_predicted_bbox,
            "num_gt_bbox": num_gt_bbox,
            "num_eval_images": len(img_ids_eval),
            "num_eval_categories": len(cat_ids_eval),
        })
        eval_summary["coco_metrics"] = coco_metrics

        # Efficiency metrics per IoU threshold
        for iou_thr in iou_thresholds:
            pr_res = compute_pr_manual(
                cocoGt.dataset["annotations"],
                remapped,
                iou_thr=iou_thr,
                image_ids_eval=img_ids_eval,
            )

            bbox_additions = pr_res["fn"]
            bbox_removals  = pr_res["fp"]
            precision_iou  = pr_res["precision"]
            recall_iou     = pr_res["recall"]

            eff_key = f"efficiency_metrics@iou_{iou_thr:.2f}"
            eff_metrics = eval_summary[eff_key]  # already created by make_empty_eval

            eff_metrics.update({
                "bbox_additions": bbox_additions,
                "bbox_removals": bbox_removals,
                "precision": precision_iou,
                "recall": recall_iou,
            })

            # Time & Cost calculations
            total_train_time = preds.get("info", {}).get("total_training_time_s", 0.0)
            total_inf_time = preds.get("info", {}).get("total_inference_time_s", 0.0)

            # Machine time
            total_machine_time = total_train_time + total_inf_time

            # Manual time
            num_initial_bbox = preds.get("info", {}).get("num_initial_bbox", 0)
            add_time = ANNOTATION_TIMES.get(dataset_name, {}).get("add", DEFAULT_ADD_TIME)
            remove_time = ANNOTATION_TIMES.get(dataset_name, {}).get("remove", DEFAULT_REMOVE_TIME)
            review_time = REVIEW_TIME_PER_BOX

            total_manual_time = (
                num_initial_bbox * add_time
                + num_predicted_bbox * review_time
                + bbox_removals * remove_time
                + bbox_additions * add_time
            )
            print(f"initialanns: {num_initial_bbox}, totalinft: {total_inf_time}, tottraintime: {total_train_time:.2f}"   )
            # Cost
            total_compute_cost = (total_machine_time / 3600) * HOURLY_COMPUTE_COST

            # Total annotation time
            total_annotation_time = total_machine_time + total_manual_time

            # Per-bbox statistics
            annotation_time_per_bbox = (
                total_annotation_time / num_gt_bbox if num_gt_bbox > 0 else 0.0
            )
            annotation_cost_per_bbox = (
                total_compute_cost / num_gt_bbox if num_gt_bbox > 0 else 0.0
            )

            # Percentages
            manual_time_pct = (
                (total_manual_time / total_annotation_time) * 100.0
                if total_annotation_time > 0 else 0.0
            )
            machine_time_pct = 100.0 - manual_time_pct

            eff_metrics["total_annotation_time_s"] = total_annotation_time
            eff_metrics["annotation_time_per_bbox_s"] = annotation_time_per_bbox
            eff_metrics["annotation_time_manual_pct"] = manual_time_pct
            eff_metrics["annotation_time_machine_pct"] = machine_time_pct
            eff_metrics["total_cost_usd"] = total_compute_cost
            eff_metrics["cost_per_bbox_usd"] = annotation_cost_per_bbox

        # Write eval_summary
        eval_json = out_root / f"{dataset_name}_{subset_name}_{model_name}_evaluation.json"
        with open(eval_json, "w", encoding="utf-8") as f:
            json.dump(eval_summary, f, indent=2)

        results.append(eval_summary)

    return results


In [7]:
evaluate_predictions(predictions_dir="../Results/Experiment_1/", iou_thresholds=EVAL_IOU_THRESHOLDS, iou_type="bbox")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
creating index...
index created!
initialanns: 0, totalinft: 9.493295295978896, tottraintime: 0.00
initialanns: 0, totalinft: 9.493295295978896, tottraintime: 0.00
initialanns: 0, totalinft: 9.493295295978896, tottraintime: 0.00
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.020
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.9

[{'info': {'timestamp': '2025-12-27 17:08:12',
   'dataset_name': 'apples',
   'subset_name': 'test',
   'model_name': 'gd_t',
   'iou_type': 'bbox',
   'num_predicted_bbox': 0,
   'num_gt_bbox': 25,
   'num_eval_images': 2,
   'num_eval_categories': 2},
  'coco_metrics': {'AP@[.50:.95]': 0.0,
   'AP@0.50': 0.0,
   'AP@0.75': 0.0,
   'AP_small': 0.0,
   'AP_medium': 0.0,
   'AP_large': 0.0,
   'AR@1': 0.0,
   'AR@10': 0.0,
   'AR@100': 0.0,
   'AR_small': 0.0,
   'AR_medium': 0.0,
   'AR_large': 0.0},
  'efficiency_metrics@iou_0.50': {'bbox_additions': 25,
   'bbox_removals': 0,
   'precision': 0.0,
   'recall': 0.0,
   'total_annotation_time_s': 145.9932952959789,
   'annotation_time_per_bbox_s': 5.8397318118391555,
   'annotation_time_manual_pct': 93.4974443334999,
   'annotation_time_machine_pct': 6.5025556665001005,
   'total_cost_usd': 0.00535529972778525,
   'cost_per_bbox_usd': 0.00021421198911140998},
  'efficiency_metrics@iou_0.70': {'bbox_additions': 25,
   'bbox_removals': 0